# 03 IQC ＋ PBRTQC：検出時間と患者リスク
誤差を人工的に入れ（2025 年、300 回、正負交互）、IQC 単独・IQC＋PBRTQC・PBRTQC 単独の検出時間と、患者リスク MaxE(Nuf) を計算します。
続けて、成分を 1 つずつ外す解析、感度分析（最終報告値、2 号機、結果の遅れ、バイアス、CVa）、試算（QC の追加、警報上限）を実行します。
所要時間の目安：デモ 4 分。実データは 15〜60 分程度。

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "pbrtqc").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.max_columns", 30, "display.width", 200)
from pbrtqc import config as C, pipeline
print(C.describe())

> **いまのモード**は上のセルの `profile=` で確認してください（`demo` = 合成データ、`real` = 実データ）。モードを変えたら、VS Code の「Restart」でカーネルを再起動してから上から実行し直します。

In [ ]:
log = pipeline.run([s[0] for s in pipeline.steps() if s[1] == "fusion"], log_name="run_log_fusion.json")

## 主な結果（Table 2 に相当）

In [ ]:
f = pd.read_csv(C.TABLES / "fusion.csv")
k1 = f[f["k"] == 1.0].pivot(index="item", columns="system", values="med_hours")
mx = f.groupby(["item", "system"])["e_nuf"].max().unstack()
cv = pd.read_csv(C.TABLES / "iqc_totalcv.csv").set_index("item")
t2 = pd.DataFrame({"sigma": cv["sigma"], "検出(h) PBRTQC単独": k1["PB:aod+even+out"], "検出(h) IQC単独": k1["IQC"],
                   "検出(h) IQC+PBRTQC": k1["aod+even+out"], "MaxE IQC単独": mx["IQC"], "MaxE IQC+PBRTQC": mx["aod+even+out"]})
display(t2.sort_values("sigma").round(2))
print("検出時間の中央値（項目間）: IQC 単独", round(k1["IQC"].median(), 2), "h → IQC+PBRTQC", round(k1["aod+even+out"].median(), 2), "h")
print("PBRTQC 単独で成立:", int((k1["PB:aod+even+out"] <= 24).sum()), "/ 20 項目")

## σ の低い 5 項目：構成ごとの MaxE(Nuf)（Table 3 に相当）

In [ ]:
low5 = ["Na", "ALB", "Cl", "Ca", "Mg"]
mx.loc[low5, ["IQC", "aod+even+out", "aod+out", "even+out", "in+out", "pooled"]].round(1)